In [ ]:
# Edit only the attached Kaggle Input paths. This notebook is Internet-OFF.
from pathlib import Path

LEGALIR_SOURCE_PATH = Path("/kaggle/input/datasets/mduy2911/legalir/train.json")
CORPUS_PATH = Path("/kaggle/input/datasets/mduy2911/legalir/selected-contexts")
DENSE_MODEL_PATH = Path("/kaggle/input/datasets/mduy2911/bge-m3-kaggle")

DENSE_MODEL_NAME = "BAAI/bge-m3"
DENSE_DECLARED_REVISION = "5617a9f61b028005a4858fdac845db406aefb181"
EXPECTED_SOURCE_SHA256 = "c39cde9e74977e350f1456e7d487aafe67d2bcbaa4fa26fcabd557fe635635b7"
EXPECTED_SPLIT_COUNTS = {"train": 4_941, "dev": 1_036, "holdout": 1_023}
EXPECTED_DOCUMENTS = 8_532
EXPECTED_FIXED_CHUNKS = 199_816

CHUNK_SIZE = 2_000
CHUNK_OVERLAP = 200
TOP_K_CHUNKS = 2_000
DOCUMENT_AGGREGATION = "sum_top_2"
EVALUATION_DEPTHS = (10, 20, 50, 100, 200)
MAX_LENGTH = 8_192
CORPUS_BATCH_SIZE = 256
QUERY_BATCH_SIZE = 64
BASELINE_MATERIAL_TOLERANCE = 1e-3
BASELINE_REFERENCE = {
    "recall": {
        10: 0.9227799227799228,
        20: 0.9497265122265123,
        50: 0.9769144144144144,
        100: 0.9819015444015444,
        200: 0.9877734877734878,
    },
    "mrr": 0.7159583402955311,
}
RESULT_PATH = Path("/kaggle/working/title_enriched_dense_retrieval_dev_results.json")

In [ ]:
import json
from collections import defaultdict
from hashlib import sha256
from math import isfinite
from time import perf_counter

import numpy as np
import torch
import torch.nn.functional as F
import transformers
from transformers import AutoModel, AutoTokenizer


def read_json(path: Path):
    try:
        with path.open(encoding="utf-8-sig") as stream:
            return json.load(stream)
    except json.JSONDecodeError as exc:
        raise ValueError(f"{path}: invalid JSON: {exc}") from exc


def load_fixed_dev(path: Path) -> tuple[dict, dict, str]:
    source_digest = sha256(path.read_bytes()).hexdigest()
    if source_digest != EXPECTED_SOURCE_SHA256:
        raise ValueError(
            "LegalIR source SHA256 mismatch: "
            f"expected {EXPECTED_SOURCE_SHA256}, got {source_digest}"
        )
    value = read_json(path)
    if not isinstance(value, dict) or not all(
        isinstance(sample, dict) for sample in value.values()
    ):
        raise ValueError(f"{path}: expected an object keyed by sample ID")
    samples = {str(sample_id): sample for sample_id, sample in value.items()}
    if len(samples) != len(value):
        raise ValueError(f"{path}: duplicate sample IDs after string canonicalization")

    split_counts = {"train": 0, "dev": 0, "holdout": 0}
    dev_ids = []
    for sample_id, sample in samples.items():
        question = sample.get("question")
        group_key = (
            question
            if isinstance(question, str)
            else f"\0fallback-sample-id:{sample_id}"
        )
        bucket = int(sha256(group_key.encode("utf-8")).hexdigest()[:8], 16) % 100
        if bucket < 70:
            split_name = "train"
        elif bucket < 85:
            split_name = "dev"
            dev_ids.append(sample_id)
        else:
            split_name = "holdout"
        split_counts[split_name] += 1

    if split_counts != EXPECTED_SPLIT_COUNTS:
        raise ValueError(
            f"fixed split count mismatch: expected {EXPECTED_SPLIT_COUNTS}, "
            f"got {split_counts}"
        )

    dev = {sample_id: samples[sample_id] for sample_id in sorted(dev_ids)}
    if len(dev) != EXPECTED_SPLIT_COUNTS["dev"]:
        raise RuntimeError("fixed DEV selection is internally inconsistent")
    for sample_id, sample in dev.items():
        if not isinstance(sample.get("question"), str):
            raise TypeError(f"DEV sample {sample_id!r}: question must be a string")
        answer = sample.get("answer")
        if not isinstance(answer, list) or not answer:
            raise ValueError(
                f"DEV sample {sample_id!r}: expected a non-empty LegalIR answer list"
            )
        if any(document_id is None for document_id in answer):
            raise ValueError(f"DEV sample {sample_id!r}: answer contains null")
        canonical_answer = [str(document_id) for document_id in answer]
        if len(canonical_answer) != len(set(canonical_answer)):
            raise ValueError(f"DEV sample {sample_id!r}: answer contains duplicate IDs")
    return dev, split_counts, source_digest


def load_corpus(path: Path) -> list[dict]:
    json_paths = sorted(
        item
        for item in path.rglob("*")
        if item.is_file() and item.suffix.lower() == ".json"
    )
    if not json_paths:
        raise ValueError(f"{path}: corpus directory contains no JSON files")
    documents = []
    for json_path in json_paths:
        value = read_json(json_path)
        values = value if isinstance(value, list) else [value]
        if not all(isinstance(document, dict) for document in values):
            raise ValueError(f"{json_path}: expected document object(s)")
        documents.extend(values)

    document_ids = []
    for document in documents:
        if document.get("id") is None:
            raise ValueError("corpus document is missing a non-null id")
        document_id = str(document["id"])
        passage = document.get("passage")
        if not isinstance(passage, str):
            raise TypeError(f"document {document_id!r}: passage must be a string")
        document_ids.append(document_id)
    if len(document_ids) != len(set(document_ids)):
        raise ValueError("corpus contains duplicate document IDs")
    if len(documents) != EXPECTED_DOCUMENTS:
        raise ValueError(
            f"expected {EXPECTED_DOCUMENTS:,} corpus documents, got {len(documents):,}"
        )
    return documents


def fixed_window_chunks(documents: list[dict]) -> list[dict]:
    if CHUNK_SIZE <= 0 or CHUNK_OVERLAP < 0 or CHUNK_OVERLAP >= CHUNK_SIZE:
        raise ValueError("invalid fixed-window chunk parameters")
    step = CHUNK_SIZE - CHUNK_OVERLAP
    chunks = []
    for document in documents:
        document_id = str(document["id"])
        source = document["passage"]
        if not source:
            continue
        for chunk_index, start in enumerate(range(0, len(source), step)):
            end = min(start + CHUNK_SIZE, len(source))
            chunks.append(
                {
                    "chunk_id": f"{document_id}:{chunk_index}",
                    "document_id": document_id,
                    "chunk_index": chunk_index,
                    "text": source[start:end],
                    "char_start": start,
                    "char_end": end,
                }
            )
            if end == len(source):
                break
    if len(chunks) != EXPECTED_FIXED_CHUNKS:
        raise ValueError(
            f"expected {EXPECTED_FIXED_CHUNKS:,} fixed-window chunks, "
            f"got {len(chunks):,}"
        )
    validate_chunk_provenance(documents, chunks)
    return chunks


def validate_chunk_provenance(documents: list[dict], chunks: list[dict]) -> dict:
    source_by_id = {str(document["id"]): document["passage"] for document in documents}
    chunk_ids = [chunk["chunk_id"] for chunk in chunks]
    if len(chunk_ids) != len(set(chunk_ids)):
        raise ValueError("chunk IDs must be unique")
    intervals_by_document = defaultdict(list)
    for chunk in chunks:
        document_id = chunk["document_id"]
        if document_id not in source_by_id:
            raise ValueError(f"chunk references unknown document {document_id!r}")
        source = source_by_id[document_id]
        start = chunk["char_start"]
        end = chunk["char_end"]
        if not (0 <= start < end <= len(source)):
            raise ValueError(f"chunk {chunk['chunk_id']!r}: invalid source offsets")
        if chunk["text"] != source[start:end]:
            raise ValueError(f"chunk {chunk['chunk_id']!r}: provenance mismatch")
        intervals_by_document[document_id].append((start, end))

    for document_id, source in source_by_id.items():
        if not source:
            continue
        intervals = sorted(intervals_by_document[document_id])
        if not intervals or intervals[0][0] != 0:
            raise ValueError(f"document {document_id!r}: source coverage does not start at 0")
        covered_end = 0
        for start, end in intervals:
            if start > covered_end:
                raise ValueError(f"document {document_id!r}: source coverage has a gap")
            covered_end = max(covered_end, end)
        if covered_end != len(source):
            raise ValueError(f"document {document_id!r}: source coverage is incomplete")
    return {
        "all_non_empty_source_text_covered": True,
        "exact_provenance": True,
        "unique_chunk_ids": True,
        "all_document_ids_resolve": True,
    }


def distribution(values: list[int]) -> dict:
    if not values:
        return {"min": None, "median": None, "p95": None, "max": None}
    array = np.asarray(values)
    return {
        "min": int(array.min()),
        "median": float(np.median(array)),
        "p95": float(np.percentile(array, 95)),
        "max": int(array.max()),
    }


def local_model_metadata(model) -> dict:
    config_commit_hash = getattr(model.config, "_commit_hash", None)
    has_config_revision = (
        isinstance(config_commit_hash, str) and bool(config_commit_hash.strip())
    )
    if not has_config_revision:
        revision_status = "declared-offline-snapshot"
        config_commit_hash = None
    elif config_commit_hash == DENSE_DECLARED_REVISION:
        revision_status = "verified-from-config"
    else:
        revision_status = "config-mismatch"
    return {
        "model_name": DENSE_MODEL_NAME,
        "declared_revision": DENSE_DECLARED_REVISION,
        "config_commit_hash": config_commit_hash,
        "revision_status": revision_status,
        "local_input_path": str(DENSE_MODEL_PATH),
    }


def load_dense_model() -> dict:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle CUDA accelerator")
    started = perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(
        DENSE_MODEL_PATH,
        local_files_only=True,
    )
    model = AutoModel.from_pretrained(
        DENSE_MODEL_PATH,
        dtype=torch.float16,
        local_files_only=True,
    )
    model.to("cuda")
    model.eval()
    metadata = local_model_metadata(model)
    if metadata["revision_status"] == "config-mismatch":
        raise RuntimeError(
            "local model config _commit_hash does not match the declared revision: "
            f"{metadata['config_commit_hash']} != {DENSE_DECLARED_REVISION}"
        )
    return {
        "tokenizer": tokenizer,
        "model": model,
        "metadata": metadata,
        "load_seconds": perf_counter() - started,
    }


def encode_normalized_cls(
    dense_model: dict,
    texts: list[str],
    batch_size: int,
    collect_token_lengths: bool = False,
) -> dict:
    if batch_size <= 0:
        raise ValueError("encoding batch size must be positive")
    if not texts:
        raise ValueError("cannot encode an empty text collection")
    tokenizer = dense_model["tokenizer"]
    model = dense_model["model"]
    embeddings = []
    token_lengths = []
    started = perf_counter()

    for batch_start in range(0, len(texts), batch_size):
        batch = texts[batch_start : batch_start + batch_size]
        if collect_token_lengths:
            untruncated = tokenizer(
                batch,
                padding=False,
                truncation=False,
                add_special_tokens=True,
                return_length=True,
            )
            token_lengths.extend(int(length) for length in untruncated["length"])
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        inputs = {name: value.to("cuda") for name, value in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs, return_dict=True)
            embedding = outputs.last_hidden_state[:, 0]
            embedding = F.normalize(embedding, p=2, dim=1)
        if embedding.ndim != 2 or not torch.isfinite(embedding).all():
            raise RuntimeError("dense encoder returned invalid normalized CLS embeddings")
        embeddings.append(embedding.detach().cpu())

    encoded = torch.cat(embeddings, dim=0)
    if encoded.shape[0] != len(texts):
        raise RuntimeError("dense encoder output count does not match input count")
    seconds = perf_counter() - started
    token_diagnostics = None
    if collect_token_lengths:
        lengths = np.asarray(token_lengths, dtype=np.int64)
        truncated_count = int(np.sum(lengths > MAX_LENGTH))
        token_diagnostics = {
            "median_tokens": float(np.median(lengths)),
            "p95_tokens": float(np.percentile(lengths, 95)),
            "max_tokens": int(lengths.max()),
            "truncated_count": truncated_count,
            "truncated_fraction": truncated_count / len(lengths),
        }
    return {
        "embeddings": encoded,
        "seconds": seconds,
        "items_per_second": len(texts) / seconds,
        "token_diagnostics": token_diagnostics,
    }


def aggregate_sum_top_2(chunk_hits: list[dict]) -> list[str]:
    grouped_scores = defaultdict(list)
    best_chunk_rank = {}
    for hit in chunk_hits:
        document_id = hit["document_id"]
        score = float(hit["score"])
        if not isfinite(score):
            raise ValueError("retrieval score must be finite")
        grouped_scores[document_id].append(score)
        best_chunk_rank[document_id] = min(
            best_chunk_rank.get(document_id, hit["rank"]), hit["rank"]
        )
    ranked = [
        (
            document_id,
            sum(sorted(scores, reverse=True)[:2]),
            best_chunk_rank[document_id],
        )
        for document_id, scores in grouped_scores.items()
    ]
    ranked.sort(key=lambda item: (-item[1], item[2], item[0]))
    document_ids = [item[0] for item in ranked]
    if len(document_ids) != len(set(document_ids)):
        raise RuntimeError("document aggregation produced duplicate IDs")
    return document_ids


def dense_rankings(
    query_embeddings: torch.Tensor,
    corpus_embeddings: torch.Tensor,
    chunks: list[dict],
    sample_ids: list[str],
) -> dict:
    if query_embeddings.shape[0] != len(sample_ids):
        raise ValueError("query embedding count does not match sample IDs")
    if corpus_embeddings.shape[0] != len(chunks):
        raise ValueError("corpus embedding count does not match chunks")
    if TOP_K_CHUNKS != 2_000:
        raise RuntimeError("TOP_K_CHUNKS must remain exactly 2000")
    if len(chunks) < TOP_K_CHUNKS:
        raise ValueError("corpus has fewer chunks than TOP_K_CHUNKS")
    started = perf_counter()
    passage_embedding = corpus_embeddings.to("cuda")
    rankings = {}

    for batch_start in range(0, len(sample_ids), QUERY_BATCH_SIZE):
        batch_ids = sample_ids[batch_start : batch_start + QUERY_BATCH_SIZE]
        query_embedding = query_embeddings[
            batch_start : batch_start + len(batch_ids)
        ].to("cuda")
        similarity = query_embedding @ passage_embedding.T
        if not torch.isfinite(similarity).all():
            raise RuntimeError("dense dot-product similarity produced non-finite scores")
        top_scores, top_indices = torch.topk(
            similarity,
            k=TOP_K_CHUNKS,
            dim=1,
            largest=True,
            sorted=True,
        )
        for row, sample_id in enumerate(batch_ids):
            raw_hits = [
                (float(score), int(index))
                for score, index in zip(
                    top_scores[row].float().cpu().tolist(),
                    top_indices[row].cpu().tolist(),
                )
            ]
            raw_hits.sort(key=lambda item: (-item[0], item[1]))
            hits = [
                {
                    "document_id": chunks[index]["document_id"],
                    "score": score,
                    "rank": rank,
                }
                for rank, (score, index) in enumerate(raw_hits, start=1)
            ]
            rankings[sample_id] = aggregate_sum_top_2(hits)

    seconds = perf_counter() - started
    del passage_embedding
    return {
        "rankings": rankings,
        "seconds": seconds,
        "queries_per_second": len(sample_ids) / seconds,
    }


def evaluate_rankings(samples: dict, rankings: dict, depths: tuple[int, ...]) -> dict:
    if set(rankings) != set(samples):
        raise ValueError("ranking IDs do not exactly match fixed DEV IDs")
    recall_values = {depth: [] for depth in depths}
    reciprocal_ranks = []
    candidate_counts = []
    for sample_id, sample in samples.items():
        gold = {str(document_id) for document_id in sample["answer"]}
        ranked = rankings[sample_id]
        if len(ranked) != len(set(ranked)):
            raise ValueError(f"sample {sample_id!r}: ranking contains duplicate IDs")
        candidate_counts.append(len(ranked))
        for depth in depths:
            effective_k = min(depth, len(ranked))
            prefix = ranked[:effective_k]
            recall_values[depth].append(len(gold.intersection(prefix)) / len(gold))
        first_rank = next(
            (
                rank
                for rank, document_id in enumerate(ranked, start=1)
                if document_id in gold
            ),
            None,
        )
        reciprocal_ranks.append(0.0 if first_rank is None else 1.0 / first_rank)

    return {
        "recall": {
            str(depth): {
                "macro_recall": float(np.mean(values)),
                "zero_recall_rate": float(np.mean(np.asarray(values) == 0)),
                "full_recall_rate": float(np.mean(np.asarray(values) == 1)),
                "queries_below_depth": int(
                    np.sum(np.asarray(candidate_counts) < depth)
                ),
            }
            for depth, values in recall_values.items()
        },
        "mrr": float(np.mean(reciprocal_ranks)),
        "candidate_document_count": {
            **distribution(candidate_counts),
            "counts_below_evaluation_depth": {
                str(depth): int(np.sum(np.asarray(candidate_counts) < depth))
                for depth in depths
            },
        },
    }


def validate_fixed_window_baseline(metrics: dict) -> dict:
    observed = {
        f"recall@{depth}": metrics["recall"][str(depth)]["macro_recall"]
        for depth in BASELINE_REFERENCE["recall"]
    }
    observed["mrr"] = metrics["mrr"]
    expected = {
        **{
            f"recall@{depth}": value
            for depth, value in BASELINE_REFERENCE["recall"].items()
        },
        "mrr": BASELINE_REFERENCE["mrr"],
    }
    deltas = {name: observed[name] - expected[name] for name in expected}
    max_abs_delta = max(abs(value) for value in deltas.values())
    summary = {
        "tolerance": BASELINE_MATERIAL_TOLERANCE,
        "max_absolute_delta": max_abs_delta,
        "passed": max_abs_delta <= BASELINE_MATERIAL_TOLERANCE,
    }
    if not summary["passed"]:
        print(json.dumps({"baseline_mismatch_deltas": deltas}, indent=2))
        raise RuntimeError(
            "fixed-window BGE-M3 baseline differs materially from the sanity reference; "
            "stop before evaluating another research axis"
        )
    return summary


def metric_deltas(alternative: dict, baseline: dict) -> dict:
    return {
        **{
            f"recall@{depth}": (
                alternative["recall"][str(depth)]["macro_recall"]
                - baseline["recall"][str(depth)]["macro_recall"]
            )
            for depth in EVALUATION_DEPTHS
        },
        "mrr": alternative["mrr"] - baseline["mrr"],
    }


def paired_coverage(
    samples: dict,
    baseline_rankings: dict,
    alternative_rankings: dict,
    baseline_label: str,
    alternative_label: str,
) -> dict:
    output = {}
    for depth in (100, 200):
        categories = {
            "gold_found_by_both": 0,
            f"gold_{baseline_label}_only": 0,
            f"gold_{alternative_label}_only": 0,
            "gold_found_by_neither": 0,
        }
        alternative_unique_queries = 0
        baseline_unique_queries = 0
        for sample_id, sample in samples.items():
            gold = {str(document_id) for document_id in sample["answer"]}
            baseline_ranked = baseline_rankings[sample_id]
            alternative_ranked = alternative_rankings[sample_id]
            baseline_prefix = set(
                baseline_ranked[: min(depth, len(baseline_ranked))]
            )
            alternative_prefix = set(
                alternative_ranked[: min(depth, len(alternative_ranked))]
            )
            categories["gold_found_by_both"] += len(
                gold & baseline_prefix & alternative_prefix
            )
            categories[f"gold_{baseline_label}_only"] += len(
                (gold & baseline_prefix) - alternative_prefix
            )
            categories[f"gold_{alternative_label}_only"] += len(
                (gold & alternative_prefix) - baseline_prefix
            )
            categories["gold_found_by_neither"] += len(
                gold - (baseline_prefix | alternative_prefix)
            )
            alternative_unique_queries += bool(
                (gold & alternative_prefix) - baseline_prefix
            )
            baseline_unique_queries += bool(
                (gold & baseline_prefix) - alternative_prefix
            )
        output[str(depth)] = {
            **categories,
            f"queries_where_{alternative_label}_recovers_at_least_one_gold_absent_from_{baseline_label}_prefix": int(
                alternative_unique_queries
            ),
            f"queries_where_{baseline_label}_recovers_at_least_one_gold_absent_from_{alternative_label}_prefix": int(
                baseline_unique_queries
            ),
        }
    return output


def embedding_metadata(encoded: dict) -> dict:
    embeddings = encoded["embeddings"]
    return {
        "number_of_embeddings": int(embeddings.shape[0]),
        "embedding_dimension": int(embeddings.shape[1]),
        "embedding_dtype": str(embeddings.dtype).removeprefix("torch."),
    }

In [ ]:
import os

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

for required_path, expected_kind in (
    (LEGALIR_SOURCE_PATH, "file"),
    (CORPUS_PATH, "directory"),
    (DENSE_MODEL_PATH, "directory"),
):
    exists = required_path.is_file() if expected_kind == "file" else required_path.is_dir()
    if not exists:
        raise FileNotFoundError(
            f"Attach the required offline {expected_kind} at: {required_path}"
        )

assert TOP_K_CHUNKS == 2_000
assert DOCUMENT_AGGREGATION == "sum_top_2"
assert MAX_LENGTH == 8_192
assert CHUNK_SIZE == 2_000 and CHUNK_OVERLAP == 200
assert EVALUATION_DEPTHS == (10, 20, 50, 100, 200)

run_started = perf_counter()
dev_samples, split_counts, source_digest = load_fixed_dev(LEGALIR_SOURCE_PATH)
documents = load_corpus(CORPUS_PATH)
chunks = fixed_window_chunks(documents)
source_by_id = {str(document["id"]): document["passage"] for document in documents}
usable_name_by_id = {
    str(document["id"]): document["name"]
    for document in documents
    if isinstance(document.get("name"), str) and bool(document["name"].strip())
}
name_diagnostics = {
    "documents_with_non_empty_name": len(usable_name_by_id),
    "documents_without_usable_name": len(documents) - len(usable_name_by_id),
}

raw_texts = [chunk["text"] for chunk in chunks]
enriched_texts = [
    (
        usable_name_by_id[chunk["document_id"]] + "\n" + chunk["text"]
        if chunk["document_id"] in usable_name_by_id
        else chunk["text"]
    )
    for chunk in chunks
]
for chunk, raw_text, enriched_text in zip(chunks, raw_texts, enriched_texts):
    source = source_by_id[chunk["document_id"]]
    if chunk["text"] != source[chunk["char_start"] : chunk["char_end"]]:
        raise RuntimeError(f"chunk {chunk['chunk_id']!r}: raw evidence was modified")
    if raw_text != chunk["text"]:
        raise RuntimeError("baseline retrieval text differs from raw chunk text")
    expected_enriched = (
        usable_name_by_id[chunk["document_id"]] + "\n" + chunk["text"]
        if chunk["document_id"] in usable_name_by_id
        else chunk["text"]
    )
    if enriched_text != expected_enriched:
        raise RuntimeError("title enrichment is not exactly name + newline + raw chunk")

print(json.dumps({"name_coverage_before_retrieval": name_diagnostics}, indent=2))
dense_model = load_dense_model()
query_encoding = encode_normalized_cls(
    dense_model,
    [sample["question"] for sample in dev_samples.values()],
    batch_size=QUERY_BATCH_SIZE,
)

torch.cuda.reset_peak_memory_stats()
baseline_encoding = encode_normalized_cls(
    dense_model,
    raw_texts,
    batch_size=CORPUS_BATCH_SIZE,
    collect_token_lengths=True,
)
baseline_embedding_info = embedding_metadata(baseline_encoding)
baseline_dense = dense_rankings(
    query_encoding["embeddings"],
    baseline_encoding["embeddings"],
    chunks,
    list(dev_samples),
)
baseline_peak_gpu_memory = int(torch.cuda.max_memory_allocated())
baseline_metrics = evaluate_rankings(
    dev_samples, baseline_dense["rankings"], EVALUATION_DEPTHS
)
baseline_sanity = validate_fixed_window_baseline(baseline_metrics)
del baseline_encoding["embeddings"]
torch.cuda.empty_cache()

torch.cuda.reset_peak_memory_stats()
enriched_encoding = encode_normalized_cls(
    dense_model,
    enriched_texts,
    batch_size=CORPUS_BATCH_SIZE,
    collect_token_lengths=True,
)
enriched_embedding_info = embedding_metadata(enriched_encoding)
enriched_dense = dense_rankings(
    query_encoding["embeddings"],
    enriched_encoding["embeddings"],
    chunks,
    list(dev_samples),
)
enriched_peak_gpu_memory = int(torch.cuda.max_memory_allocated())
enriched_metrics = evaluate_rankings(
    dev_samples, enriched_dense["rankings"], EVALUATION_DEPTHS
)
deltas = metric_deltas(enriched_metrics, baseline_metrics)
coverage = paired_coverage(
    dev_samples,
    baseline_dense["rankings"],
    enriched_dense["rankings"],
    "baseline",
    "enriched",
)

result = {
    "experiment": "title-enriched retrieval text versus raw fixed-window text",
    "split": {
        "name": "fixed DEV",
        "queries": len(dev_samples),
        "verified_partition_counts": split_counts,
        "source_sha256": source_digest,
        "evaluated_partition": "dev only",
    },
    "controls": {
        "documents": len(documents),
        "chunks": len(chunks),
        "same_chunk_objects_and_ids_for_both_representations": True,
        "chunk_size": CHUNK_SIZE,
        "overlap": CHUNK_OVERLAP,
        "step": CHUNK_SIZE - CHUNK_OVERLAP,
        "top_k_chunks": TOP_K_CHUNKS,
        "document_aggregation": "sum top-2 retrieval chunk scores",
        "evaluation_depths": list(EVALUATION_DEPTHS),
        "no_query_instruction": True,
        "no_cross_encoder": True,
        "no_article_aware_chunking": True,
        "no_fusion": True,
    },
    "dense_model": {
        **dense_model["metadata"],
        "representation": "L2-normalized CLS hidden state",
        "similarity": "query_embedding @ passage_embedding.T",
        "max_length": MAX_LENGTH,
        "dynamic_padding": True,
        "device": "cuda",
        "dtype": "float16",
    },
    "name_coverage": name_diagnostics,
    "provenance_checks": {
        "chunk_boundaries_unchanged": True,
        "chunk_ids_unchanged": True,
        "raw_chunk_text_unchanged": True,
        "name_affects_retrieval_text_only": True,
        "missing_name_falls_back_to_raw_chunk": True,
    },
    "baseline_raw_text": {
        "metrics": baseline_metrics,
        "embedding": baseline_embedding_info,
        "token_diagnostics": baseline_encoding["token_diagnostics"],
        "baseline_sanity": baseline_sanity,
    },
    "title_enriched_text": {
        "semantics": "source name + newline + raw chunk; raw chunk only when name is unusable",
        "metrics": enriched_metrics,
        "embedding": enriched_embedding_info,
        "token_diagnostics": enriched_encoding["token_diagnostics"],
    },
    "title_enriched_minus_baseline": deltas,
    "paired_coverage": coverage,
    "fusion_performed": False,
    "interpretation_scope": (
        "Tests whether document name adds dense retrieval context beyond passage text; "
        "it does not test LegalQA generation input."
    ),
    "runtime": {
        "model_load_seconds": dense_model["load_seconds"],
        "query_encoding_seconds": query_encoding["seconds"],
        "query_encoding_queries_per_second": query_encoding["items_per_second"],
        "baseline_raw_text": {
            "number_of_chunks": baseline_embedding_info["number_of_embeddings"],
            "corpus_encoding_seconds": baseline_encoding["seconds"],
            "chunks_per_second": baseline_encoding["items_per_second"],
            "retrieval_seconds": baseline_dense["seconds"],
            "queries_per_second": baseline_dense["queries_per_second"],
            "peak_gpu_memory_bytes": baseline_peak_gpu_memory,
        },
        "title_enriched_text": {
            "number_of_chunks": enriched_embedding_info["number_of_embeddings"],
            "corpus_encoding_seconds": enriched_encoding["seconds"],
            "chunks_per_second": enriched_encoding["items_per_second"],
            "retrieval_seconds": enriched_dense["seconds"],
            "queries_per_second": enriched_dense["queries_per_second"],
            "peak_gpu_memory_bytes": enriched_peak_gpu_memory,
        },
        "corpus_batch_size": CORPUS_BATCH_SIZE,
        "query_batch_size": QUERY_BATCH_SIZE,
        "total_seconds": perf_counter() - run_started,
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
    },
}

RESULT_PATH.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({
    "baseline_metrics": baseline_metrics,
    "enriched_metrics": enriched_metrics,
    "deltas": deltas,
    "token_diagnostics": {
        "baseline": baseline_encoding["token_diagnostics"],
        "enriched": enriched_encoding["token_diagnostics"],
    },
    "paired_coverage": coverage,
    "runtime": result["runtime"],
}, ensure_ascii=False, indent=2))
print("Saved aggregate-only results:", RESULT_PATH)